# 종목 하나로 보는 이벤트 전략

`MTD` 한 종목으로 RSI 반전 전략(예제 `rsi-reversal`)을 따라간다.
RSI 14가 30 아래로 **들어가는 날** 진입하고, 손절 10%·익절 15%·보유 20일 중 먼저 오는 것에 청산한다.
`SYMBOL`만 바꾸면 다른 종목으로 같은 경로를 볼 수 있다.

In [ ]:
import sys
from dataclasses import fields, is_dataclass
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ai_studio").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def table(value):
    """프레임·전략 판단·이력 튜플을 표로."""
    rows = getattr(value, "rows", None) or getattr(value, "signals", None) or getattr(value, "positions", None)
    if rows is None:
        rows = value if isinstance(value, (list, tuple)) else (value,)
    if not rows:
        return pd.DataFrame()
    if is_dataclass(rows[0]):
        return pd.DataFrame([{f.name: getattr(row, f.name) for f in fields(rows[0])} for row in rows])
    return pd.DataFrame(rows)

from ai_studio.n03_market_data import bars_until, history_from_bars, load_ohlc
from ai_studio.n12_strategies import example_rsi_reversal
from ai_studio.n14_engine import ExecutionContext, execute, execution_order
from ai_studio.n15_ohlc_backtest import RunSettings, run_graph_backtest

SYMBOL = "MTD"

## 1. 시세를 한 종목만 남긴다

In [ ]:
bars = tuple(bar for bar in load_ohlc() if bar.symbol == SYMBOL)
if not bars:
    raise ValueError(f"{SYMBOL} 일봉이 없습니다.")
history = history_from_bars(bars)
days = [bar.timestamp for bar in history[SYMBOL]]
print(f"{SYMBOL}  거래일 {len(days)}  기간 {days[0]:%Y-%m-%d} ~ {days[-1]:%Y-%m-%d}")
table(history[SYMBOL][:3])

## 2. 그래프

`rsi-oversold` 프리셋(RSI → 임계 조건)의 출력을 상태 전이에 넣어 진입 이벤트를 만들고,
손절·보유기간 이벤트를 청산 포트에 함께 연결한다. 청산 포트는 여러 이벤트를 합친다.

In [ ]:
graph = example_rsi_reversal()
pd.DataFrame([{"노드": n.id, "블록": n.function_id, "파라미터": n.params} for n in execution_order(graph)])

## 3. 날마다 조건과 이벤트

거래일마다 한 번씩 실행해 RSI, 과매도 여부, 진입 이벤트를 모은다. 진입은 거짓 → 참으로 바뀐 날만 생긴다.

In [ ]:
rows = []
for day in days[60:]:
    out = execute(graph, ExecutionContext(day, history))
    rsi = out.port("rsi_rsi", "value").rows
    low = out.port("rsi_low", "condition").rows
    rows.append({
        "date": f"{day:%Y-%m-%d}", "close": bars_until(history, SYMBOL, day)[-1].close,
        "rsi": round(rsi[0].value, 1) if rsi else None,
        "oversold": low[0].matched if low else None,
        "entry": bool(out.port("enter", "entry").rows),
    })
daily = pd.DataFrame(rows)
daily[daily["entry"] | daily["oversold"].fillna(False)].head(20)

## 4. 백테스트와 매매 내역

손절·보유기간 청산은 보유 상태가 있어야 나온다. 원장이 날마다 보유를 컨텍스트로 넘긴다.
신호는 그날 종가, 체결은 다음 거래일 시가다.

진입 비중은 1/최대 보유 수(10)이고 수량은 온주다. 투자금이 1만 달러면 한 종목 몫이 1천 달러라
주가가 1천 달러가 넘는 MTD는 1주도 살 수 없다. 그래서 10만 달러로 돌린다.

In [ ]:
book = run_graph_backtest(graph, history, RunSettings(signal_frequency="daily", initial_capital=100_000,
                                                     fee_rate=0.001, slippage_rate=0.0005))
trades = table(book.trade_log)
trades[["symbol", "side", "quantity", "signal_at", "filled_at", "price", "fee", "realized_pnl"]] if not trades.empty else "체결 없음"

In [ ]:
curve = table(book.portfolio_history)
last = book.portfolio_history[-1]
print(f"최종 {last.total_value:,.2f}  수익률 {last.cumulative_return:+.2%}  최대낙폭 {curve['drawdown'].min():+.2%}")
curve[["as_of", "cash", "securities_value", "total_value", "drawdown"]].tail(10)